# Import des bibliothèques nécessaires

In [1]:
from pathlib import Path
import pandas as pd
import re



# Chargement des données

In [2]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_phrases.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Tiens! Clotilde, finit-il par dire, tu recopie..."
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Dans sa longue blouse noire, elle était très g..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,Des chaises et des fauteuils antiques traînaie...
4,1893_20_Le_docteur_Pascal._clean.txt,4,"Mais il dut prendre une chaise, la planche du ..."


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21260 entries, 0 to 21259
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nom_fichier     21260 non-null  str  
 1   id_paquet       21260 non-null  int64
 2   phrases_paquet  21260 non-null  str  
dtypes: int64(1), str(2)
memory usage: 498.4 KB


# Préparation du dataframe 

In [4]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [5]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [6]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [7]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [8]:
df["nb_mots"] = df["texte"].str.split().str.len()

In [9]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_mots"
]]

In [10]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [11]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [12]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",214
1,1865 La confession de Claude.,1865,1,2,"La mansarde entière me réclame les rires, les ...",187
2,1865 La confession de Claude.,1865,1,3,Le grillon chantait; le souffle harmonieux des...,146
3,1865 La confession de Claude.,1865,1,4,"brunes et rieuses filles, étaient reines des m...",190
4,1865 La confession de Claude.,1865,1,5,"Pars cependant, puisque tu as soif de la vie. ...",126


In [13]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")